In [4]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

True

In [5]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [6]:
len(INDEX_START)

186

# Load Local Database

In [7]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

70

In [8]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-04-09


# Update Existing Tickers

In [9]:
today = "2025-04-28"

for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]
    start = (data.index[-1] + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue
    
    INDEX_DATA[ticker] = pd.concat([data, new_data])
    INDEX_DATA[ticker].dropna(inplace = True)

100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [00:50<00:00,  1.38it/s]


In [10]:
INDEX_DATA['000300.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-04-22,12.2931,3.5380,3783.9522
2025-04-23,12.2687,3.5279,3786.8824
2025-04-24,12.2906,3.5176,3784.3574
2025-04-25,12.2609,3.5798,3786.9937
2025-04-28,12.3204,3.5752,3781.6188


# Download New Tickers

In [11]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

 70%|████████████████████████████████████████████████████████▎                       | 131/186 [00:36<00:15,  3.55it/s]


KeyError: 'PE_TTM'

In [13]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [12]:
len(INDEX_DATA)

131